In [ ]:
# # XGBoost for regression, by hand
# We grow XGBoost's regression tree from scratch with NumPy on four students (CGPA and package in LPA),
# check every similarity score, gain and output value from the Note, then compare with the real XGBoost library
# and with scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import GradientBoostingRegressor

## 1. The data and the first model (the mean)

In [ ]:
# four students: CGPA (input) and package in LPA (output)
df = pd.DataFrame({"cgpa": [6.7, 9.0, 7.5, 5.0], "package": [4.5, 11.0, 6.0, 8.0]})
x, y = df["cgpa"].to_numpy(), df["package"].to_numpy()

# stage 1: the base model predicts the mean for everyone
f0 = y.mean()
df["pred1"] = f0
# residuals: actual minus predicted
df["res1"] = y - f0
print("mean:", f0)
df

## 2. Similarity score, gain and output value

In [ ]:
def similarity(r, lam=0.0):
    # (sum of residuals)^2 / (number of residuals + lambda)
    return r.sum() ** 2 / (len(r) + lam)

def output(r, lam=0.0):
    # sum of residuals / (number of residuals + lambda)
    return r.sum() / (len(r) + lam)

r = df["res1"].to_numpy()
print("root similarity:", similarity(r))

# candidate thresholds: midpoints of the sorted CGPA values
xs = np.sort(x)
thresholds = (xs[1:] + xs[:-1]) / 2
for t in thresholds:
    left, right = r[x < t], r[x >= t]
    gain = similarity(left) + similarity(right) - similarity(r)
    print(f"CGPA < {t:.2f}: S_L={similarity(left):.4f}  S_R={similarity(right):.4f}  gain={gain:.4f}")

## 3. A from-scratch XGBoost regression tree (exact greedy, lambda, gamma, max_depth)

In [ ]:
def grow(x, r, lam=0.0, gamma=0.0, depth=0, max_depth=2):
    """Return a nested dict: a leaf {'out': w} or a split {'t', 'gain', 'left', 'right'}."""
    node = {"res": r, "sim": similarity(r, lam), "out": output(r, lam)}
    if depth == max_depth or len(r) < 2:
        return node
    # try every midpoint and keep the largest gain
    xs = np.unique(x)
    best = None
    for t in (xs[1:] + xs[:-1]) / 2:
        m = x < t
        gain = similarity(r[m], lam) + similarity(r[~m], lam) - similarity(r, lam)
        if best is None or gain > best[1]:
            best = (t, gain)
    t, gain = best
    m = x < t
    node.update(t=t, gain=gain,
                left=grow(x[m], r[m], lam, gamma, depth + 1, max_depth),
                right=grow(x[~m], r[~m], lam, gamma, depth + 1, max_depth))
    # pruning, bottom up: a split whose children are leaves goes if gain - gamma < 0
    if "t" not in node["left"] and "t" not in node["right"] and gain - gamma < 0:
        for k in ("t", "gain", "left", "right"):
            node.pop(k)
    return node

def show(node, indent=""):
    if "t" in node:
        print(f"{indent}CGPA < {node['t']:.2f}  (similarity {node['sim']:.2f}, gain {node['gain']:.2f})")
        show(node["left"], indent + "    ")
        show(node["right"], indent + "    ")
    else:
        print(f"{indent}leaf: residuals {np.round(node['res'], 3)}, similarity {node['sim']:.2f}, output {node['out']:.4f}")

def predict(node, v):
    while "t" in node:
        node = node["left"] if v < node["t"] else node["right"]
    return node["out"]

tree1 = grow(x, r)
show(tree1)

## 4. Stage 2: mean + eta x tree output

In [ ]:
eta = 0.3
df["pred2"] = f0 + eta * np.array([predict(tree1, v) for v in x])
df["res2"] = y - df["pred2"]
df.round(4)

## 5. Check: scikit-learn gives the same stage-2 predictions
With lambda = 0 the gain is the drop in squared error, so the XGBoost tree equals a squared-error regression tree
grown on the residuals; GradientBoostingRegressor starts from the mean and adds learning_rate x tree.

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=1, learning_rate=0.3, max_depth=2).fit(x.reshape(-1, 1), y)
print("sklearn thresholds:", gbr.estimators_[0, 0].tree_.threshold[gbr.estimators_[0, 0].tree_.feature >= 0])
print("sklearn:", gbr.predict(x.reshape(-1, 1)))
print("ours:   ", df["pred2"].to_numpy())
assert np.allclose(gbr.predict(x.reshape(-1, 1)), df["pred2"])

## 6. Check: the real XGBoost library
We set every setting to match the hand calculation. The dump shows each split's gain and each leaf's value;
XGBoost stores leaf values already multiplied by eta (0.3 x 0.625 = 0.1875).

In [ ]:
X = x.reshape(-1, 1)
for lam, gamma in [(0, 0), (1, 0), (0, 6)]:
    model = xgb.XGBRegressor(n_estimators=1, max_depth=2, learning_rate=0.3, reg_lambda=lam, gamma=gamma,
                             min_child_weight=0, base_score=f0, tree_method="exact", device="cpu",
                             random_state=42).fit(X, y)
    print(f"lambda={lam}, gamma={gamma}: predictions {model.predict(X)}")
    print(model.get_booster().get_dump(with_stats=True)[0])
# the lambda = 0 predictions are our stage 2
model = xgb.XGBRegressor(n_estimators=1, max_depth=2, learning_rate=0.3, reg_lambda=0, min_child_weight=0,
                         base_score=f0, tree_method="exact", device="cpu").fit(X, y)
assert np.allclose(model.predict(X), df["pred2"], atol=1e-5)

## 7. The second tree, on the stage-2 residuals

In [ ]:
tree2 = grow(x, df["res2"].to_numpy())
show(tree2)
df["pred3"] = df["pred2"] + eta * np.array([predict(tree2, v) for v in x])
df["res3"] = y - df["pred3"]
df.round(4)

## 8. Extra: lambda shrinks similarity scores and outputs

In [ ]:
for lam in (0, 1):
    print(f"lambda = {lam}")
    show(grow(x, r, lam=lam))

## 9. Extra: gamma prunes weak splits

In [ ]:
for gamma in (0, 6, 20):
    print(f"gamma = {gamma}")
    show(grow(x, r, gamma=gamma))

In [ ]:
# two trees in XGBoost match our stage 3
model2 = xgb.XGBRegressor(n_estimators=2, max_depth=2, learning_rate=0.3, reg_lambda=0, min_child_weight=0,
                          base_score=f0, tree_method="exact", device="cpu").fit(X, y)
print(model2.predict(X), df["pred3"].to_numpy())
assert np.allclose(model2.predict(X), df["pred3"], atol=1e-5)